In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

ROOT = Path(r"E:\TESIS MAESTRIA\Desarrollo_clustering_maestria")

PATH_MATCH = ROOT / "02_data_cleaning/outputs/match_final_empresas.csv"
PATH_AUDIT = ROOT / "02_data_cleaning/outputs/audit_df.csv"

OUT_REVISION = ROOT / "02_data_cleaning/outputs/revision_manual_elegibilidad_empresas.csv"
OUT_RESUMEN = ROOT / "02_data_cleaning/outputs/resumen_revision_elegibilidad_empresas.csv"

def normalizar_texto(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().upper()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalizar_columna(c):
    s = str(c).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalizar_ruc(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(13)
    )

def encontrar_leads_xlsx(root):
    candidatos = []
    for nombre in ["leads.xlsx", "Leads.xlsx", "LEADS.xlsx", "leads(3).xlsx"]:
        candidatos.extend(list(root.rglob(nombre)))

    candidatos = [
        p for p in candidatos
        if ".ipynb_checkpoints" not in str(p)
        and "__pycache__" not in str(p)
        and "venv" not in str(p).lower()
    ]

    if not candidatos:
        raise FileNotFoundError(
            "No se encontró leads.xlsx dentro del proyecto. "
            "Define manualmente PATH_LEADS."
        )

    return candidatos[0]

print("=== 1) CARGA DE match_final_empresas.csv ===")

match = pd.read_csv(PATH_MATCH, dtype={"RUC": str})
match["RUC"] = normalizar_ruc(match["RUC"])

for col in ["source_label", "name_raw", "name_norm", "source_winner", "verdict"]:
    if col in match.columns:
        match[col] = match[col].fillna("").astype(str)

if "score" in match.columns:
    match["score"] = pd.to_numeric(match["score"], errors="coerce")

match["name_norm_std"] = match["name_norm"].apply(normalizar_texto)

print(f"match_final shape: {match.shape}")
print(f"RUC únicos: {match['RUC'].nunique()}")
print(f"Duplicados RUC: {match['RUC'].duplicated().sum()}")

print("\n=== 2) CARGA DE audit_df.csv ===")

if PATH_AUDIT.exists():
    audit = pd.read_csv(PATH_AUDIT, dtype={"ruc": str})
    audit["ruc"] = normalizar_ruc(audit["ruc"])

    for col in ["source_norm", "candidate_norm", "source_label", "verdict", "reason"]:
        if col in audit.columns:
            audit[col] = audit[col].fillna("").astype(str)

    if "score" in audit.columns:
        audit["score"] = pd.to_numeric(audit["score"], errors="coerce")

    if "confidence" in audit.columns:
        audit["confidence"] = pd.to_numeric(audit["confidence"], errors="coerce")

    audit["source_norm_std"] = audit["source_norm"].apply(normalizar_texto)

    audit_min = audit[
        [
            "ruc",
            "source_norm_std",
            "candidate_norm",
            "source_label",
            "score",
            "verdict",
            "confidence",
            "reason"
        ]
    ].copy()

    audit_min = audit_min.rename(
        columns={
            "ruc": "RUC",
            "source_label": "audit_source_label",
            "score": "audit_score",
            "verdict": "audit_verdict",
            "confidence": "audit_confidence",
            "reason": "audit_reason",
        }
    )

    audit_min = (
        audit_min
        .sort_values(["RUC", "source_norm_std", "audit_score"], ascending=[True, True, False])
        .drop_duplicates(["RUC", "source_norm_std"], keep="first")
        .reset_index(drop=True)
    )

    print(f"audit_df shape: {audit.shape}")
    print(f"audit_min shape: {audit_min.shape}")
else:
    audit_min = pd.DataFrame(
        columns=[
            "RUC",
            "source_norm_std",
            "candidate_norm",
            "audit_source_label",
            "audit_score",
            "audit_verdict",
            "audit_confidence",
            "audit_reason",
        ]
    )
    print("ADVERTENCIA: No existe audit_df.csv. Se continuará sin candidate_norm.")

revision = match.merge(
    audit_min,
    left_on=["RUC", "name_norm_std"],
    right_on=["RUC", "source_norm_std"],
    how="left"
)

for col in [
    "candidate_norm",
    "audit_source_label",
    "audit_verdict",
    "audit_reason",
]:
    if col in revision.columns:
        revision[col] = revision[col].fillna("").astype(str)

print(f"revision shape tras merge audit: {revision.shape}")

print("\n=== 3) CARGA DE leads.xlsx Y DETECCIÓN DE PAÍS ===")

PATH_LEADS = encontrar_leads_xlsx(ROOT)
print(f"leads.xlsx encontrado en: {PATH_LEADS}")

leads = pd.read_excel(PATH_LEADS, dtype=str)
leads.columns = [str(c).strip() for c in leads.columns]

cols_norm = {normalizar_columna(c): c for c in leads.columns}

print("\nColumnas detectadas:")
for k, v in cols_norm.items():
    print(f"{k} -> {v}")

col_empresa = None
for posible in ["company", "empresa", "nombre empresa", "compania"]:
    if posible in cols_norm:
        col_empresa = cols_norm[posible]
        break

col_pais = None
for posible in ["pais", "country"]:
    if posible in cols_norm:
        col_pais = cols_norm[posible]
        break

if col_empresa is None:
    raise ValueError(
        f"No se encontró columna de empresa en leads.xlsx. "
        f"Columnas: {leads.columns.tolist()}"
    )

if col_pais is None:
    raise ValueError(
        f"No se encontró columna país/country en leads.xlsx. "
        f"Columnas: {leads.columns.tolist()}"
    )

print(f"\nColumna empresa detectada: {col_empresa}")
print(f"Columna país detectada: {col_pais}")

leads["name_norm_std"] = leads[col_empresa].apply(normalizar_texto)
leads["pais_leads"] = leads[col_pais].fillna("").astype(str).str.strip()
leads["pais_leads_std"] = leads["pais_leads"].apply(normalizar_texto)

leads_pais = (
    leads[["name_norm_std", "pais_leads", "pais_leads_std"]]
    .drop_duplicates("name_norm_std")
)

revision = revision.merge(
    leads_pais,
    on="name_norm_std",
    how="left"
)

revision["pais_leads"] = revision["pais_leads"].fillna("")
revision["pais_leads_std"] = revision["pais_leads_std"].fillna("")

print(f"revision shape tras merge leads: {revision.shape}")

print("\n=== 4) FLAGS DE REVISIÓN ===")

revision["flag_pais_internacional"] = (
    revision["pais_leads_std"].ne("")
    & ~revision["pais_leads_std"].isin({"ECUADOR"})
)

keywords_sospecha_internacional = [
    " MEXICO ",
    " GUYANA ",
    " COLOMBIA ",
    " PERU ",
    " CHILE ",
    " BRASIL ",
    " VENEZUELA ",
    " ARGENTINA ",
    " COSTA RICA ",
    " PANAMA ",
    " NICARAGUA ",
    " S A DE C V ",
    " SA DE CV ",
    " C V ",
    " SAB ",
    " PA NI ",
]

def tiene_sospecha_internacional_nombre(nombre):
    s = f" {normalizar_texto(nombre)} "
    return any(k in s for k in keywords_sospecha_internacional)

revision["flag_sospecha_internacional_nombre"] = revision["name_norm"].apply(
    tiene_sospecha_internacional_nombre
)

genericos = {
    "GRUPO",
    "GLOBAL",
    "PHARMA",
    "BUSINESS",
    "FORUM",
    "CREDITO",
    "MEXICANA",
    "EXPRESS",
    "LAB",
    "BIO",
    "AMP",
    "CORPORACION",
    "CENTER",
    "CONSULTING",
    "SOLUCIONES",
    "GROUP",
    "MEXICO",
    "ECUADOR",
    "COMERCIAL",
    "INDUSTRIAL",
}

def es_match_generico(row):
    cand = normalizar_texto(row.get("candidate_norm", ""))
    source_winner = normalizar_texto(row.get("source_winner", ""))

    if cand in genericos:
        return True

    if len(cand.split()) <= 1 and source_winner in {"SRI FANTASIA", "SRI RAZON"}:
        return True

    return False

revision["flag_match_generico"] = revision.apply(es_match_generico, axis=1)

conteo_ruc = (
    revision
    .groupby("RUC")["name_norm"]
    .nunique()
    .reset_index(name="nombres_por_ruc")
)

nombres_por_ruc = (
    revision
    .groupby("RUC")["name_norm"]
    .apply(lambda x: " | ".join(sorted(set(x.astype(str)))))
    .reset_index(name="nombres_asociados_al_mismo_ruc")
)

revision = revision.merge(conteo_ruc, on="RUC", how="left")
revision = revision.merge(nombres_por_ruc, on="RUC", how="left")

revision["flag_ruc_duplicado"] = revision["nombres_por_ruc"].fillna(0).astype(int) > 1

print("Flags creados correctamente.")

print("\n=== 5) DECISIÓN SUGERIDA CONSERVADORA ===")

def sugerir_decision(row):
    motivos = []

    if row["flag_pais_internacional"]:
        motivos.append(f"País en leads.xlsx = {row['pais_leads']}")
        return "EXCLUIR_INTERNACIONAL", "; ".join(motivos)

    if row["flag_match_generico"]:
        motivos.append("Match candidato genérico o de baja calidad")
        return "EXCLUIR_MATCH_GENERICO", "; ".join(motivos)

    if row["flag_sospecha_internacional_nombre"]:
        motivos.append("Nombre contiene señal de posible operación internacional")
        return "REVISION_MANUAL", "; ".join(motivos)

    if row["flag_ruc_duplicado"]:
        motivos.append(f"RUC compartido por {int(row['nombres_por_ruc'])} nombres")
        return "REVISION_MANUAL", "; ".join(motivos)

    verdict = str(row.get("verdict", "")).lower().strip()
    audit_verdict = str(row.get("audit_verdict", "")).lower().strip()

    if verdict == "uncertain" or audit_verdict == "uncertain":
        return "REVISION_MANUAL", "Veredicto LLM uncertain"

    if verdict == "incorrect" or audit_verdict == "incorrect":
        return "REVISION_MANUAL", "Veredicto LLM incorrect; revisar antes de excluir"

    ruc = str(row.get("RUC", "")).strip()
    if len(ruc) != 13 or not ruc.isdigit():
        return "REVISION_MANUAL", "RUC no válido; revisar manualmente"

    return "ELEGIBLE_ECUADOR", "Sin alertas relevantes; elegible provisional"

revision[["decision_sugerida", "motivo_sugerido"]] = revision.apply(
    lambda row: pd.Series(sugerir_decision(row)),
    axis=1
)

revision["decision_manual"] = ""
revision["comentario_manual"] = ""

revision["google_query"] = (
    revision["name_raw"].astype(str)
    + " "
    + revision["RUC"].astype(str)
    + " Ecuador empresa"
)

print("Decisión sugerida creada.")

print("\n=== 6) EXPORTAR CSV DE REVISIÓN ===")

cols_prioridad = [
    "source_label",
    "name_raw",
    "name_norm",
    "RUC",
    "source_winner",
    "score",
    "verdict",
    "pais_leads",
    "candidate_norm",
    "audit_reason",
    "flag_pais_internacional",
    "flag_sospecha_internacional_nombre",
    "flag_match_generico",
    "flag_ruc_duplicado",
    "nombres_por_ruc",
    "nombres_asociados_al_mismo_ruc",
    "decision_sugerida",
    "motivo_sugerido",
    "decision_manual",
    "comentario_manual",
    "google_query",
]

cols_existentes = [c for c in cols_prioridad if c in revision.columns]
otros = [c for c in revision.columns if c not in cols_existentes]

revision_out = revision[cols_existentes + otros].copy()

OUT_REVISION.parent.mkdir(parents=True, exist_ok=True)
revision_out.to_csv(OUT_REVISION, index=False, encoding="utf-8-sig")

resumen = (
    revision_out
    .groupby("decision_sugerida")
    .agg(
        filas=("RUC", "count"),
        ruc_unicos=("RUC", "nunique"),
        con_pais_leads=("pais_leads", lambda x: (x.astype(str).str.strip() != "").sum()),
        ruc_duplicados=("flag_ruc_duplicado", "sum"),
        match_generico=("flag_match_generico", "sum"),
        sospecha_internacional_nombre=("flag_sospecha_internacional_nombre", "sum"),
        pais_internacional=("flag_pais_internacional", "sum"),
    )
    .reset_index()
)

resumen.to_csv(OUT_RESUMEN, index=False, encoding="utf-8-sig")

print(f"Exportado: {OUT_REVISION}")
print(f"Exportado: {OUT_RESUMEN}")

print("\n=== RESUMEN DECISIÓN SUGERIDA ===")
display(resumen)

print("\n=== CASOS CON PAÍS INTERNACIONAL DESDE leads.xlsx ===")
display(
    revision_out[revision_out["flag_pais_internacional"]]
    [cols_existentes]
    .head(100)
)

print("\n=== CASOS CON SOSPECHA INTERNACIONAL POR NOMBRE ===")
display(
    revision_out[revision_out["flag_sospecha_internacional_nombre"]]
    [cols_existentes]
    .head(100)
)

print("\n=== CASOS CON RUC DUPLICADO ===")
display(
    revision_out[revision_out["flag_ruc_duplicado"]]
    [cols_existentes]
    .sort_values("RUC")
    .head(100)
)

print("\n=== CASOS CON MATCH GENÉRICO ===")
display(
    revision_out[revision_out["flag_match_generico"]]
    [cols_existentes]
    .head(100)
)

print("\n=== CASOS SUGERIDOS PARA REVISIÓN MANUAL ===")
display(
    revision_out[revision_out["decision_sugerida"].eq("REVISION_MANUAL")]
    [cols_existentes]
    .head(100)
)

=== 1) CARGA DE match_final_empresas.csv ===
match_final shape: (169, 8)
RUC únicos: 159
Duplicados RUC: 10

=== 2) CARGA DE audit_df.csv ===
audit_df shape: (404, 11)
audit_min shape: (404, 8)
revision shape tras merge audit: (169, 15)

=== 3) CARGA DE leads.xlsx Y DETECCIÓN DE PAÍS ===
leads.xlsx encontrado en: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\leads.xlsx

Columnas detectadas:
company -> Company
first name -> First Name
last name -> Last Name
email -> Email
phone -> Phone
mobile -> Mobile
website -> Website
lead source -> Lead Source
industry -> Industry
no of employees -> No. of Employees
annual revenue -> Annual Revenue
interes -> Interés
cargo -> Cargo
pais -> País.
linkedin -> Linkedin

Columna empresa detectada: Company
Columna país detectada: País.
revision shape tras merge leads: (169, 17)

=== 4) FLAGS DE REVISIÓN ===
Flags creados correctamente.

=== 5) DECISIÓN SUGERIDA CONSERVADORA ===
Decisión sugerida creada.

=== 6) EXPORTAR CSV DE REVISIÓN ===
Exportado:

,decision_sugerida,filas,ruc_unicos,con_pais_leads,ruc_duplicados,match_generico,sospecha_internacional_nombre,pais_internacional
0,ELEGIBLE_ECUADOR,106,106,2,0,0,0,0
1,EXCLUIR_INTERNACIONAL,5,5,5,0,0,0,5
2,EXCLUIR_MATCH_GENERICO,54,45,0,13,54,7,0
3,REVISION_MANUAL,4,3,0,2,0,4,0



=== CASOS CON PAÍS INTERNACIONAL DESDE leads.xlsx ===


,source_label,name_raw,name_norm,RUC,source_winner,score,verdict,pais_leads,candidate_norm,audit_reason,...,flag_sospecha_internacional_nombre,flag_match_generico,flag_ruc_duplicado,nombres_por_ruc,nombres_asociados_al_mismo_ruc,decision_sugerida,motivo_sugerido,decision_manual,comentario_manual,google_query
17,LEADS,Carval,CARVAL,0993107859001,SCVS,100.0,correct,México||Colombia||Ecuador,CARVAL ECUADOR CARVALECSA,Coherencia total entre source_raw y candidate_...,...,False,False,False,1,CARVAL,EXCLUIR_INTERNACIONAL,País en leads.xlsx = México||Colombia||Ecuador,,,Carval 0993107859001 Ecuador empresa
22,LEADS,Chronos,CHRONOS,1709821787001,SRI_RAZON,100.0,correct,México,BILBAO NUNEZ CESAR CHRONOS,Coherencia total entre source_raw y candidate_...,...,False,False,False,1,CHRONOS,EXCLUIR_INTERNACIONAL,País en leads.xlsx = México,,,Chronos 1709821787001 Ecuador empresa
47,LEADS,EUROFARMA,EUROFARMA,1792377749001,SCVS_EXACT,100.0,correct,Perú,,,...,False,False,False,1,EUROFARMA,EXCLUIR_INTERNACIONAL,País en leads.xlsx = Perú,,,EUROFARMA 1792377749001 Ecuador empresa
58,LEADS,GLP,GLP,1391934829001,SCVS,100.0,correct,Brasil,AUTOGAS SYSTEMS GLP GNV S,Coherencia total entre source_raw y candidate_...,...,False,False,False,1,GLP,EXCLUIR_INTERNACIONAL,País en leads.xlsx = Brasil,,,GLP 1391934829001 Ecuador empresa
127,LEADS,Petroli,PETROLI,0991362096001,SCVS_EXACT,100.0,correct,México,,,...,False,False,False,1,PETROLI,EXCLUIR_INTERNACIONAL,País en leads.xlsx = México,,,Petroli 0991362096001 Ecuador empresa



=== CASOS CON SOSPECHA INTERNACIONAL POR NOMBRE ===


...

,source_label,name_raw,name_norm,RUC,source_winner,score,verdict,pais_leads,candidate_norm,audit_reason,...,flag_sospecha_internacional_nombre,flag_match_generico,flag_ruc_duplicado,nombres_por_ruc,nombres_asociados_al_mismo_ruc,decision_sugerida,motivo_sugerido,decision_manual,comentario_manual,google_query
59,LEADS,GNP SEGUROS,GNP SEGUROS,0106644651001,SRI_FANTASIA,100.0,correct,,SEGUROS,Coherencia total entre source_raw y candidate_...,...,False,True,True,2,GNP SEGUROS | PRIMERO SEGUROS,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GNP SEGUROS 0106644651001 Ecuador empresa
130,LEADS,PRIMERO SEGUROS,PRIMERO SEGUROS,0106644651001,SRI_FANTASIA,100.0,correct,,SEGUROS,Coherencia total entre source_raw y candidate_...,...,False,True,True,2,GNP SEGUROS | PRIMERO SEGUROS,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,PRIMERO SEGUROS 0106644651001 Ecuador empresa
113,LEADS,MTWA SOLUCIONES INTEGRALES,MTWA SOLUCIONES INTEGRALES,0190351084001,SRI_RAZON,100.0,correct,,SOLUCIONES,Coherencia total entre source_raw y candidate_...,...,False,True,True,2,MTWA SOLUCIONES INTEGRALES | SOLUCIONES INTEGR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,MTWA SOLUCIONES INTEGRALES 0190351084001 Ecuad...
149,LEADS,SOLUCIONES INTEGRALES IKNELIA,SOLUCIONES INTEGRALES IKNELIA,0190351084001,SRI_RAZON,100.0,correct,,SOLUCIONES,Coherencia total entre source_raw y candidate_...,...,False,True,True,2,MTWA SOLUCIONES INTEGRALES | SOLUCIONES INTEGR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,SOLUCIONES INTEGRALES IKNELIA 0190351084001 Ec...
68,LEADS,GRUPO CARSO,GRUPO CARSO,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,False,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO CARSO 0913220786001 Ecuador empresa
72,LEADS,GRUPO SID,GRUPO SID,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,False,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO SID 0913220786001 Ecuador empresa
74,LEADS,GRUPO TMM,GRUPO TMM,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,False,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO TMM 0913220786001 Ecuador empresa
75,LEADS,GRUPO TRIMEX S.A.,GRUPO TRIMEX,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,False,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO TRIMEX S.A. 0913220786001 Ecuador empresa
76,LEADS,GRUPO VASCONIA SAB,GRUPO VASCONIA SAB,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,True,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO VASCONIA SAB 0913220786001 Ecuador empresa
77,LEADS,GRUPO ZUCARMEX,GRUPO ZUCARMEX,0913220786001,SRI_FANTASIA,100.0,correct,,GRUPO,Coherencia total entre source_raw y candidate_...,...,False,True,True,6,GRUPO CARSO | GRUPO SID | GRUPO TMM | GRUPO TR...,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GRUPO ZUCARMEX 0913220786001 Ecuador empresa



=== CASOS CON MATCH GENÉRICO ===


,source_label,name_raw,name_norm,RUC,source_winner,score,verdict,pais_leads,candidate_norm,audit_reason,...,flag_sospecha_internacional_nombre,flag_match_generico,flag_ruc_duplicado,nombres_por_ruc,nombres_asociados_al_mismo_ruc,decision_sugerida,motivo_sugerido,decision_manual,comentario_manual,google_query
14,LEADS,BIIS LOGISTICS,BIIS LOGISTICS,1712571007001,SRI_FANTASIA,100.000000,correct,,LOGISTICS,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,BIIS LOGISTICS,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,BIIS LOGISTICS 1712571007001 Ecuador empresa
15,LEADS,BRINSA S.A.,BRINSA,0991349073001,SRI_RAZON,100.000000,correct,,BRINSA,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,BRINSA,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,BRINSA S.A. 0991349073001 Ecuador empresa
19,LEADS,CEMEX,CEMEX,1702311349001,SRI_FANTASIA,100.000000,correct,,CEMEX,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,CEMEX,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,CEMEX 1702311349001 Ecuador empresa
24,LEADS,CID / GRUPO KNOBLOCH,CID GRUPO KNOBLOCH,1702889617001,SRI_FANTASIA,100.000000,correct,,CID,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,CID GRUPO KNOBLOCH,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,CID / GRUPO KNOBLOCH 1702889617001 Ecuador emp...
30,LEADS,CONAGRA FOODS,CONAGRA FOODS,1002503223001,SRI_FANTASIA,100.000000,correct,,FOODS,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,CONAGRA FOODS,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,CONAGRA FOODS 1002503223001 Ecuador empresa
39,LEADS,DANONE,DANONE,0990973962001,SRI_RAZON,90.909088,correct,,DANON,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,DANONE,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,DANONE 0990973962001 Ecuador empresa
42,LEADS,Econofarm S.A Corporación GPF,ECONOFARM CORPORACION GPF,1791715772001,SRI_RAZON,100.000000,correct,,ECONOFARM,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,ECONOFARM CORPORACION GPF,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,Econofarm S.A Corporación GPF 1791715772001 Ec...
43,LEADS,ELI LILLY Y CIA DE MÉXICO S.A. DE CV,ELI LILLY MEXICO CV,1791714113001,SRI_RAZON,100.000000,correct,,ELI,Coherencia total entre source_raw y candidate_...,...,True,True,False,1,ELI LILLY MEXICO CV,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,ELI LILLY Y CIA DE MÉXICO S.A. DE CV 179171411...
52,HORAS,Farmapiel,FARMAPIEL,0992830964001,SRI_FANTASIA,100.000000,correct,,FARMAPIEL,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,FARMAPIEL,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,Farmapiel 0992830964001 Ecuador empresa
57,LEADS,GLOBAL HITSS,GLOBAL HITSS,1792923611001,SRI_RAZON,100.000000,correct,,GLOBAL,Coherencia total entre source_raw y candidate_...,...,False,True,False,1,GLOBAL HITSS,EXCLUIR_MATCH_GENERICO,Match candidato genérico o de baja calidad,,,GLOBAL HITSS 1792923611001 Ecuador empresa



=== CASOS SUGERIDOS PARA REVISIÓN MANUAL ===


,source_label,name_raw,name_norm,RUC,source_winner,score,verdict,pais_leads,candidate_norm,audit_reason,...,flag_sospecha_internacional_nombre,flag_match_generico,flag_ruc_duplicado,nombres_por_ruc,nombres_asociados_al_mismo_ruc,decision_sugerida,motivo_sugerido,decision_manual,comentario_manual,google_query
0,LEADS,7-ELEVEN MEXICO,7 ELEVEN MEXICO,1400612998001,SRI_FANTASIA,100.0,correct,,7 ELEVEN,Coherencia total entre source_raw y candidate_...,...,True,False,False,1,7 ELEVEN MEXICO,REVISION_MANUAL,Nombre contiene señal de posible operación int...,,,7-ELEVEN MEXICO 1400612998001 Ecuador empresa
25,LEADS,"CLAYTON DE MÉXICO, S.A. DE C.V.",CLAYTON MEXICO C V,1792424585001,SRI_RAZON,100.0,correct,,C C,Coherencia total entre source_raw y candidate_...,...,True,False,True,2,CLAYTON MEXICO C V | NEOLPHARMA C V,REVISION_MANUAL,Nombre contiene señal de posible operación int...,,,"CLAYTON DE MÉXICO, S.A. DE C.V. 1792424585001 ..."
95,LEADS,LEONALI S. DE R. L. DE C.V.,LEONALI S R L C V,1091797620001,SCVS,100.0,correct,,L S,Coherencia total entre source_raw y candidate_...,...,True,False,False,1,LEONALI S R L C V,REVISION_MANUAL,Nombre contiene señal de posible operación int...,,,LEONALI S. DE R. L. DE C.V. 1091797620001 Ecua...
116,LEADS,NEOLPHARMA S.A. DE C.V.,NEOLPHARMA C V,1792424585001,SRI_RAZON,100.0,correct,,C C,Coherencia total entre source_raw y candidate_...,...,True,False,True,2,CLAYTON MEXICO C V | NEOLPHARMA C V,REVISION_MANUAL,Nombre contiene señal de posible operación int...,,,NEOLPHARMA S.A. DE C.V. 1792424585001 Ecuador ...


In [3]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

ROOT = Path(r"E:\TESIS MAESTRIA\Desarrollo_clustering_maestria")

PATH_LEADS = ROOT / "leads.xlsx"
PATH_MATCH = ROOT / "02_data_cleaning/outputs/match_final_empresas.csv"

OUT_DIAG = ROOT / "02_data_cleaning/outputs/diagnostico_leads_internacionales_vs_match_final.csv"

def normalizar_texto(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().upper()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalizar_columna(c):
    s = str(c).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalizar_ruc(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(13)
    )

leads = pd.read_excel(PATH_LEADS, dtype=str)
leads.columns = [str(c).strip() for c in leads.columns]

cols_norm = {normalizar_columna(c): c for c in leads.columns}

col_empresa = cols_norm.get("company")
col_pais = cols_norm.get("pais")

if col_empresa is None:
    raise ValueError(f"No se encontró columna Company. Columnas: {leads.columns.tolist()}")

if col_pais is None:
    raise ValueError(f"No se encontró columna País. Columnas: {leads.columns.tolist()}")

leads["name_norm_std"] = leads[col_empresa].apply(normalizar_texto)
leads["pais_raw"] = leads[col_pais].fillna("").astype(str).str.strip()
leads["pais_std"] = leads["pais_raw"].apply(normalizar_texto)

leads_con_pais = leads[leads["pais_std"].ne("")].copy()

leads_con_pais["pais_contiene_ecuador"] = leads_con_pais["pais_std"].str.contains("ECUADOR", na=False)
leads_con_pais["pais_solo_ecuador"] = leads_con_pais["pais_std"].eq("ECUADOR")
leads_con_pais["pais_internacional_o_mixto"] = (
    leads_con_pais["pais_std"].ne("")
    & ~leads_con_pais["pais_solo_ecuador"]
)

match = pd.read_csv(PATH_MATCH, dtype={"RUC": str})
match["RUC"] = normalizar_ruc(match["RUC"])
match["name_norm_std"] = match["name_norm"].apply(normalizar_texto)

cols_match = [
    "source_label",
    "name_raw",
    "name_norm",
    "RUC",
    "source_winner",
    "score",
    "verdict"
]

diag = leads_con_pais.merge(
    match[cols_match + ["name_norm_std"]],
    on="name_norm_std",
    how="left",
    indicator=True
)

diag["esta_en_match_final"] = diag["_merge"].eq("both")

def clasificar_pais(row):
    if row["pais_solo_ecuador"]:
        return "PAIS_ECUADOR"
    if row["pais_contiene_ecuador"]:
        return "PAIS_MIXTO_INCLUYE_ECUADOR"
    return "PAIS_INTERNACIONAL"

diag["clasificacion_pais"] = diag.apply(clasificar_pais, axis=1)

diag["decision_sugerida_por_pais"] = diag["clasificacion_pais"].map({
    "PAIS_ECUADOR": "ELEGIBLE_ECUADOR",
    "PAIS_MIXTO_INCLUYE_ECUADOR": "REVISION_MANUAL",
    "PAIS_INTERNACIONAL": "EXCLUIR_INTERNACIONAL"
})

cols_out = [
    col_empresa,
    "pais_raw",
    "pais_std",
    "clasificacion_pais",
    "decision_sugerida_por_pais",
    "esta_en_match_final",
    "source_label",
    "name_raw",
    "name_norm",
    "RUC",
    "source_winner",
    "score",
    "verdict"
]

diag_out = diag[cols_out].copy()

diag_out.to_csv(OUT_DIAG, index=False, encoding="utf-8-sig")

print("Exportado:", OUT_DIAG)

print("\n=== Resumen por país/clasificación ===")
display(
    diag_out.groupby(["clasificacion_pais", "esta_en_match_final"])
    .size()
    .reset_index(name="conteo")
)

print("\n=== Leads con país internacional o mixto ===")
display(diag_out)

print("\n=== Internacionales del Excel que NO están en match_final ===")
display(
    diag_out[
        diag_out["clasificacion_pais"].isin(["PAIS_INTERNACIONAL", "PAIS_MIXTO_INCLUYE_ECUADOR"])
        & ~diag_out["esta_en_match_final"]
    ]
)

Exportado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\diagnostico_leads_internacionales_vs_match_final.csv

=== Resumen por país/clasificación ===


,clasificacion_pais,esta_en_match_final,conteo
0,PAIS_ECUADOR,True,2
1,PAIS_INTERNACIONAL,False,16
2,PAIS_INTERNACIONAL,True,6
3,PAIS_MIXTO_INCLUYE_ECUADOR,True,1



=== Leads con país internacional o mixto ===


,Company,pais_raw,pais_std,clasificacion_pais,decision_sugerida_por_pais,esta_en_match_final,source_label,name_raw,name_norm,RUC,source_winner,score,verdict
0,Coppel,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Home Interiors,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Carval,México||Colombia||Ecuador,MEXICO COLOMBIA ECUADOR,PAIS_MIXTO_INCLUYE_ECUADOR,REVISION_MANUAL,True,LEADS,Carval,CARVAL,0993107859001,SCVS,100.0,correct
3,Corona Atizapan,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Grupo Ron Diplomático,Venezuela,VENEZUELA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Novaventa,Colombia,COLOMBIA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,EUROFARMA,Perú,PERU,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,True,LEADS,EUROFARMA,EUROFARMA,1792377749001,SCVS_EXACT,100.0,correct
7,GLP,Brasil,BRASIL,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,True,LEADS,GLP,GLP,1391934829001,SCVS,100.0,correct
8,NovaThinka,Colombia,COLOMBIA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Kellog,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== Internacionales del Excel que NO están en match_final ===


,Company,pais_raw,pais_std,clasificacion_pais,decision_sugerida_por_pais,esta_en_match_final,source_label,name_raw,name_norm,RUC,source_winner,score,verdict
0,Coppel,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Home Interiors,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Corona Atizapan,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Grupo Ron Diplomático,Venezuela,VENEZUELA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Novaventa,Colombia,COLOMBIA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NovaThinka,Colombia,COLOMBIA,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Kellog,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Kellog,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,GRUPO GALERÍA,México,MEXICO,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,Arcor,Perú,PERU,PAIS_INTERNACIONAL,EXCLUIR_INTERNACIONAL,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
